# Prediction analysis

Explore saved predictions from `experiments/results/<split>/<experiment>/<run>/`.
Run all cells using the project environment (`uv sync`), from the project root
or any directory inside the repository. No models are trained here.

Select a split, experiment, and numbered run. Dates and the plot use **target
time in UTC**, with an inclusive end date. Values are displayed in **g CO₂/kWh**.
MAE, RMSE, and R² are loaded from the saved run and always describe its **full
split**, even when the plotted date range is narrower. Parameters are shown
alongside the metrics. The initial view covers the first seven calendar days.
Rerun the notebook after creating new experiment results.

In [ ]:
import json
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output, display

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "docs/initial_assignment.md").exists()
)
RESULTS_DIR = ROOT / "experiments/results"
run_paths = {
    (split, path.parent.parent.name, path.parent.name): path.parent
    for split in ("valid", "test")
    for path in sorted((RESULTS_DIR / split).glob("*/*/predictions.csv"))
}
if not run_paths:
    raise FileNotFoundError("No predictions found. Run an experiment first.")

print(f"Found {len(run_paths)} saved runs.")

## Metrics by split

Compare all saved runs separately for validation and test, sorted by ascending
MAE (g CO₂/kWh). Metrics cover the full split and do not depend on plot selections.
Tables read run files directly; no metrics collection step is needed.
Compare runs with matching datasets, targets, and evaluation periods.

In [ ]:
metric_rows = []
for (split, experiment, run), folder in run_paths.items():
    metrics = json.loads((folder / "metrics.json").read_text())
    metric_rows.append({
        "Split": split,
        "Experiment": experiment,
        "Run": run,
        "Model": metrics["model"],
        "Target": metrics["target"],
        "MAE (g CO₂/kWh)": metrics["mae_g_co2_per_kwh"],
        "RMSE (g CO₂/kWh)": metrics["rmse_g_co2_per_kwh"],
        "R²": metrics["r2"],
        "Parameters": json.dumps(json.loads((folder / "params.json").read_text()), sort_keys=True),
    })
all_metrics = pd.DataFrame(metric_rows)
for split in ("valid", "test"):
    print(f"{split.upper()} — sorted by MAE (ascending)")
    split_metrics = (
        all_metrics.loc[all_metrics["Split"] == split]
        .drop(columns="Split")
        .sort_values("MAE (g CO₂/kWh)", kind="stable")
        .reset_index(drop=True)
    )
    if split_metrics.empty:
        print("No saved runs for this split.")
    else:
        with pd.option_context("display.max_rows", None):
            display(split_metrics.round(3))

## Select predictions

Changing the split or experiment updates the available runs. Changing the run
resets the date range to its first week. Missing observations remain gaps;
predictions are not resampled or interpolated.

In [ ]:
split_selector = widgets.Dropdown(
    description="Split:",
    options=[split for split in ("valid", "test") if any(k[0] == split for k in run_paths)],
)
experiment_selector = widgets.Dropdown(
    description="Experiment:", layout=widgets.Layout(width="450px"),
)
run_selector = widgets.Dropdown(description="Run:")
start_date = widgets.DatePicker(description="Start date:")
end_date = widgets.DatePicker(description="End date:")
output = widgets.Output()
state = {"loading": False}


def render(change=None):
    if state["loading"]:
        return
    with output:
        clear_output(wait=True)
        metrics = state["metrics"]
        display(pd.DataFrame([{
            "Split": metrics["split"],
            "Predictions (full split)": metrics["n_predictions"],
            "MAE (g CO₂/kWh)": metrics["mae_g_co2_per_kwh"],
            "RMSE (g CO₂/kWh)": metrics["rmse_g_co2_per_kwh"],
            "R²": metrics["r2"],
        }]).round(3))
        print(f"Target: {metrics['target']} | Target offset: {metrics['target_offset_hours']} h")
        print("Parameters:", json.dumps(state["params"], sort_keys=True))
        if start_date.value is None or end_date.value is None:
            print("Choose both a start date and an end date.")
            return
        if start_date.value > end_date.value:
            print("Start date must be on or before end date.")
            return
        lower = pd.Timestamp(start_date.value, tz="UTC")
        upper = pd.Timestamp(end_date.value, tz="UTC") + pd.Timedelta(days=1)
        frame = state["predictions"]
        selected = frame.loc[(frame.index >= lower) & (frame.index < upper)]
        if selected.empty:
            print("No predictions available for this date range.")
            return
        print(f"Showing {len(selected):,} of {len(frame):,} predictions.")
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(selected.index, selected["y"] * 1000, label="Actual", linewidth=1)
        ax.plot(selected.index, selected["y_pred"] * 1000, label="Forecast", linewidth=1)
        ax.set(
            title=f"{split_selector.value} / {experiment_selector.value} / {run_selector.value}",
            xlabel="Target time (UTC)", ylabel="Carbon intensity (g CO₂/kWh)",
        )
        ax.legend()
        ax.grid(True, alpha=0.25)
        fig.autofmt_xdate()
        fig.tight_layout()
        plt.show()
        plt.close(fig)


def load_selection(change):
    if state["loading"]:
        return
    state["loading"] = True
    try:
        if change["owner"] is split_selector:
            experiment_selector.options = sorted({
                experiment for split, experiment, run in run_paths
                if split == split_selector.value
            })
            experiment_selector.value = experiment_selector.options[0]
        if change["owner"] in (split_selector, experiment_selector):
            run_selector.options = sorted(
                run for split, experiment, run in run_paths
                if split == split_selector.value and experiment == experiment_selector.value
            )
            run_selector.value = run_selector.options[0]
        folder = run_paths[(
            split_selector.value, experiment_selector.value, run_selector.value,
        )]
        frame = pd.read_csv(folder / "predictions.csv")
        frame["target_time (UTC)"] = pd.to_datetime(frame["target_time (UTC)"], utc=True)
        state["predictions"] = frame.set_index("target_time (UTC)").sort_index()
        state["metrics"] = json.loads((folder / "metrics.json").read_text())
        state["params"] = json.loads((folder / "params.json").read_text())
        first = state["predictions"].index.min().date()
        last = state["predictions"].index.max().date()
        for picker in (start_date, end_date):
            picker.value = None
            picker.min = None
            picker.max = None
            picker.min = first
            picker.max = last
        start_date.value = first
        end_date.value = min(last, first + pd.Timedelta(days=6))
    finally:
        state["loading"] = False
    render()


for selector in (split_selector, experiment_selector, run_selector):
    selector.observe(load_selection, names="value")
for picker in (start_date, end_date):
    picker.observe(render, names="value")

load_selection({"owner": split_selector})
display(widgets.VBox([
    widgets.HBox([split_selector, experiment_selector, run_selector]),
    widgets.HBox([start_date, end_date]),
    output,
]))